# Семинар 2. Алгоритмы и структуры данных

In [ ]:
!pip install line_profiler -q

In [ ]:
import random
import time

random.seed(42)

## Поиск "медленных" частей алгоритма

Так как реализация всех алгоритмов так или иначе была на лекциях, на семинаре мы простые способы, как можно анализировать ваш код, чтобы находить узкие места в плане времени выполнения.

Пусть у нас есть очень плохая функция:

In [ ]:
def slow_search_function(texts, query, min_len=5):
    query_words = query.lower().split()

    filtered_texts = []
    for t in texts:
        if len(t) > min_len:
            filtered_texts.append(t)

    scores = []
    for txt in filtered_texts:
        txt_words = txt.lower().split()
        cnt = 0
        for qw in query_words:
            for tw in txt_words:
                if qw == tw:
                    cnt += 1
        scores.append(cnt)

    indexed = list(enumerate(scores))
    n = len(indexed)
    for i in range(n):
        for j in range(0, n-i-1):
            if indexed[j][1] < indexed[j+1][1]:
                indexed[j], indexed[j+1] = indexed[j+1], indexed[j]

    best_idx = indexed[0][0]
    return best_idx, filtered_texts[best_idx]

Давайте сначала обсудим, что в ней сделано плохо.

А теперь попробуем запустить. Для этого сгенерируем случайные данные, по которым будем искать

In [ ]:
n = 5000
base = ["уютно чисто быстро вкусно", "дорого вкусно центр", "чисто дешево парк",
        "уютно вкусно быстро", "центр парк чисто", "быстро вкусно уютно",
        "дорого центр вкусно", "тихо красиво дешево", "быстро удобно чисто"]
texts = [' '.join(random.sample(base, k=i % 3 + 1)) for i in range(n)]
texts[:3]

In [ ]:
query = "уютно чисто вкусно"
slow_search_function(texts, query, min_len=5)

Есть ощущение, что работает очень медленное, хотя задача простейшая. Чтобы перейти от ощущений к реальной оценке, попробуем замерить время выполнения.

Универсальный и самый простой метод - библиотека time.

In [ ]:
start = time.time()
idx, text = slow_search_function(texts, query)
end = time.time()

print(f"Лучший индекс: {idx}, текст: {text}")
print(f"Общее время: {end - start:.3f} сек")

Одноко если вы работаете в юпитере, то удобно применять магические команды:

In [ ]:
%time idx, text = slow_search_function(texts, query)

In [ ]:
%%time
idx, text = slow_search_function(texts, query)

Мы также можем стат. значимо оценить время работы с помощью команды timeit. Это бывает полезно, если ваша функция зависит от случайных параметров.

In [ ]:
%%timeit
idx, text = slow_search_function(texts, query)

Тепреь мы знаем, что функция правда работает медленно. Однако, если бы мы не знали, какие именно части там работают плохо, то по имеющимся оценкам никак не смогли бы этого понять. Для более подробного анализа можно использовать команду prun:

In [ ]:
%prun -s cumulative slow_search_function(texts, query)

Ничего непонятно :(

На самом деле, так происходит из-за того, что даная команда предназначена для анализа сложного кода со множеством вложенных функций. А у нас вся логика в одной функции, и все отдельные ее шаги достаточно быстрые.

Для таких случаев есть комада lprun, расширение для нее мы устанавливали выше


In [ ]:
%load_ext line_profiler

%lprun -f slow_search_function slow_search_function(texts, query)

Однако давайте все-таки попробуем добиться чего-то полезного от prun. Для этого разделим нашу функцию на части

In [ ]:
def filter_texts(texts, min_len):
    filtered_texts = []
    for t in texts:
        if len(t) > min_len:
            filtered_texts.append(t)
    return filtered_texts

def count_scores(filtered_texts, query_words):
    scores = []
    for txt in filtered_texts:
        txt_words = txt.lower().split()
        cnt = 0
        for qw in query_words:
            for tw in txt_words:
                if qw == tw:
                    cnt += 1
        scores.append(cnt)
    return scores

def sort_results(scores):
    indexed = list(enumerate(scores))
    n = len(indexed)
    for i in range(n):
        for j in range(0, n-i-1):
            if indexed[j][1] < indexed[j+1][1]:
                indexed[j], indexed[j+1] = indexed[j+1], indexed[j]
    return indexed


def slow_search_function_splited(texts, query, min_len=5):
    query_words = query.lower().split()

    filtered_texts = filter_texts(texts, min_len)
    scores = count_scores(filtered_texts, query_words)
    indexed = sort_results(scores)

    best_idx = indexed[0][0]
    return best_idx, filtered_texts[best_idx]

In [ ]:
%prun -s cumulative slow_search_function_splited(texts, query)

In [ ]:
%lprun -f slow_search_function_splited slow_search_function_splited(texts, query)

А теперь попробуем ускорить какую-нибудь часть. Самое простое - сортировка, так как она есть встроенная в питон

In [ ]:
def sort_results(scores):
    indexed = list(enumerate(scores))
    indexed.sort(key=lambda x: x[1], reverse=True)
    return indexed

In [ ]:
%lprun -f slow_search_function_splited slow_search_function_splited(texts, query)

Стало заметно быстрее!

## Задание

Вам дан очень плохой и медленный код, который в списке отелей ищет наиболее подходящий. Ваша задача - проанализировать, какие его части приводят к такой медленной работе, и постараться их ускорить.

Что ожидается в результате:
- Анализ частей кода: скорость работы, какие алгоритмы используются;
- Исправления в коде: оценк будет ставиться по внесенным улучшениям.

Работаем в командах. В последние десять минут пары я спрошу несколько случайных человек рассказать, что у вас получилось.

Код, с которым вы будете работать, можно скачать вот так:

In [ ]:
!wget https://raw.githubusercontent.com/klyshinsky/Math4Linguists2026/refs/heads/main/Seminar_2/bad_hotel_search.py

Функция, которая создает нам случайные отели для тестирования

In [ ]:
def generate_hotels(n):
    phrases = [
        "nice pool", "air conditioning", "free wifi", "breakfast included",
        "spacious room", "great view", "close to metro", "supermarket near",
        "quiet area", "family friendly", "luxury spa", "fitness center",
        "budget option", "clean and safe", "parking available", "pet friendly",
        "24 hour reception", "room service", "beach front", "mountain view"
    ]
    hotels = []
    for i in range(n):
        desc = " ".join(random.sample(phrases, k=random.randint(2, 4)))
        hotels.append({
            'id': i,
            'name': f"Hotel_{i}",
            'stars': random.randint(1, 5),
            'x': random.uniform(0, 100),
            'y': random.uniform(0, 100),
            'rating': round(random.uniform(0, 10), 1),
            'price': random.randint(50, 500),
            'description': desc
        })
    return hotels

In [ ]:
n = 5000
hotels = generate_hotels(n)
hotels[0]

Пример запуска

In [ ]:
from bad_hotel_search import find_top_hotels

In [ ]:
x = 50.0
y = 50.0
max_price = 200
min_rating = 7.0
category = 3
qtext = "pool wifi"

find_top_hotels(hotels, x, y, max_price, min_rating, category, qtext)